# First API call: transcribe speech with Whisper through Aitta

In this notebook you send recordings to a Whisper model running on CSC's
**Aitta** service, through its **OpenAI-compatible API**, and compare the
model's transcript with the correct text.

The recordings come from the FLEURS dataset prepared with the course scripts
(`run-prepare-data-lumi.sh`). Each entry in the prepared data contains:

| Column | Contents |
|---|---|
| `id` | a unique name for the recording, e.g. `test-000002-1736` |
| `audio` | the recording as a complete WAV file |
| `text` | the correct transcript |
| `duration` | length in seconds |

## 1. Imports

`openai` is the client library for OpenAI-compatible APIs; Aitta speaks the same
API. `datasets` reads the prepared data.

In [1]:
import glob
import os
from getpass import getpass, getuser

import openai
from datasets import load_from_disk
from IPython.display import Audio

## 2. Settings

In [2]:
AITTA_BASE_URL = "https://aitta-api.csc.fi/openai/v1"   # Aitta's OpenAI-compatible API
ASR_MODEL = "openai/whisper-large-v3"                     # the speech-to-text model on Aitta

DATASET = "fleurs-fi"   # data name from run-prepare-data-lumi.sh ("fleurs-en" for English)
SPLIT = "test"          # "train", "validation" or "test"

# Where the prepared data is. None = find it on LUMI automatically.
# On your own computer, give the folder that contains data/<DATASET>, e.g. "~/asr"
WORK_DIR = None

## 3. Open the prepared data

In [3]:
# Find the folder where the course scripts saved the data:
# /scratch/<project>/<username>/asr on LUMI, or WORK_DIR if you set it above.
if WORK_DIR is None:
    found = glob.glob(f"/scratch/project_*/{getuser()}/asr")
    if not found:
        raise FileNotFoundError("No prepared data found. Prepare it first with "
                                "run-prepare-data-lumi.sh, or set WORK_DIR above.")
    WORK_DIR = found[0]
WORK_DIR = os.path.expanduser(WORK_DIR)

# Opening the dataset is instant: each entry is read from disk only when used
ds = load_from_disk(os.path.join(WORK_DIR, "data", DATASET, SPLIT))
print(f"{len(ds)} recordings in {DATASET} ({SPLIT})")

916 recordings in fleurs-fi (test)


## 4. Look at and listen to one recording

`ds[number]` gives one entry. Choose a recording by its position (0, 1, 2, …).

In [4]:
entry = 0                 # change this to look at other recordings

example = ds[entry]
print(f"Id       : {example['id']}")
print(f"Duration : {example['duration']:.1f} s")
print(f"Text     : {example['text']}")

Audio(example["audio"])   # a player for the recording

Id       : test-000000-1862
Duration : 18.6 s
Text     : Yhdysvaltojen ilmailuviranomainen väittää, että NextGen on järjestelmä, jonka ansiosta lentokoneet voivat lentää lyhyempiä reittejä, säästää vuosittain miljoonia gallonia polttoainetta ja vähentää hiilidioksidipäästöjä.


## 5. Connect to Aitta

The access token is read from the environment variable `AITTA_API_KEY` if it is
set; otherwise you are asked to type it. It is not shown or saved in the notebook.

In [5]:
token = os.environ.get("AITTA_API_KEY") or getpass("Aitta access token: ")

# base_url tells the client to talk to Aitta instead of OpenAI
client = openai.OpenAI(api_key=token, base_url=AITTA_BASE_URL)

Aitta access token:  ········


## 6. Transcribe one recording

The function sends one recording to the model and prints the model's transcript
next to the correct text. Each call to it is **one API call**.

- The `audio` column already holds a complete WAV file, so it is sent as it is,
  as `(file name, file contents, file type)`.
- `response_format="verbose_json"` also returns the language the model detected.

In [6]:
def transcribe_entry(entry):
    """Transcribe one recording. entry: a position (0, 1, ...) or an id such as "test-000002-1736"."""
    # Find the recording by position or by id
    index = entry if isinstance(entry, int) else ds["id"].index(entry)
    example = ds[index]

    # One API call: send the WAV file to the model
    transcription = client.audio.transcriptions.create(
        file=("audio.wav", example["audio"], "audio/wav"),
        model=ASR_MODEL,
        response_format="verbose_json",
    )

    print(f"Recording : {example['id']} ({example['duration']:.1f} s)")
    print(f"Language  : {transcription.language}")
    print(f"Correct   : {example['text']}")
    print(f"Model     : {transcription.text.strip()}")

In [12]:
result = transcribe_entry(entry)   # the recording chosen in section 4

Recording : test-000000-1862 (18.6 s)
Language  : fi
Correct   : Yhdysvaltojen ilmailuviranomainen väittää, että NextGen on järjestelmä, jonka ansiosta lentokoneet voivat lentää lyhyempiä reittejä, säästää vuosittain miljoonia gallonia polttoainetta ja vähentää hiilidioksidipäästöjä.
Model     : Yhdysvaltojen ilmailuviranomainen väittää, että NextGen on järjestelmä, jonka ansiosta lentokoneet voivat lentää lyhyempiä reittejä, säästää vuosittain miljoonia kalonia polttoainetta ja vähentää hiilidioksidipäästöjä.


**Look at:** where do the correct text and the model's transcript differ? Small
differences, such as one letter in a long word, are common in Finnish.


## Summary

- The prepared data keeps each recording as a complete WAV file, so a recording
  can be sent to the API directly, with its correct text next to it.
- Aitta provides Whisper through the OpenAI-compatible API, so the standard
  `openai` client works by changing `base_url`.
- One recording is one API call. The next notebook transcribes many recordings
  and calculates how many words the model got wrong.